# Capítulo 12. Redes neuronales

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Redes neuronales artificiales

La formulación matemática de **álgebra lineal, cálculo, descenso por gradiente y redes neuronales** se desarrolla con mayor profundidad
en los capítulos 2, 4 y 16 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos de aprendizaje

Al finalizar este capítulo podrás:

- explicar la estructura básica de una neurona artificial;
- interpretar entradas, pesos, sesgo y funciones de activación;
- describir el funcionamiento de una red neuronal multicapa;
- comprender la propagación hacia adelante y la retropropagación;
- distinguir entre clasificación binaria y multiclase;
- preparar correctamente los datos para una red neuronal;
- implementar una red sencilla desde cero en R;
- entrenar modelos con `neuralnet`;
- construir una red moderna con `keras`;
- evaluar el desempeño mediante matriz de confusión y métricas;
- reconocer sobreajuste, regularización y errores frecuentes.

## Introducción

Las **redes neuronales artificiales** son modelos computacionales inspirados, de forma muy simplificada, en la manera en que las neuronas biológicas reciben, transforman y transmiten señales.

Una red neuronal puede aprender relaciones complejas entre variables de entrada y una variable de respuesta. Esto la hace útil para:

- clasificación;
- regresión;
- reconocimiento de imágenes;
- procesamiento de señales;
- detección de patrones;
- predicción de series;
- aplicaciones biomédicas y ambientales.

La idea central es construir muchas unidades simples, llamadas **neuronas artificiales**, conectadas entre sí mediante pesos ajustables.

## La neurona artificial

Una neurona recibe varias entradas:

$$
x_1,x_2,\ldots,x_p
$$

Cada entrada tiene un peso:

$$
w_1,w_2,\ldots,w_p
$$

La neurona calcula una combinación lineal:

$$
z=w_1x_1+w_2x_2+\cdots+w_px_p+b
$$

donde $b$ es el **sesgo**.

Después aplica una función de activación:

$$
a=f(z)
$$

El resultado $a$ se transmite a otras neuronas o se utiliza como salida.

### Interpretación

- Las entradas son las características del problema.
- Los pesos indican la importancia de cada entrada.
- El sesgo permite desplazar la frontera de decisión.
- La función de activación introduce no linealidad.
- La salida representa una predicción o una señal intermedia.

## Ejemplo manual de una neurona

Supongamos que las entradas, los pesos y el sesgo son:

$$
\begin{aligned}
x_1 &= 0.8, & x_2 &= 0.4,\\
w_1 &= 1.2, & w_2 &= -0.7, & b &= 0.1.
\end{aligned}
$$

La combinación lineal se obtiene sustituyendo los valores:

$$
\begin{aligned}
z &= w_1x_1+w_2x_2+b\\
  &= (1.2)(0.8)+(-0.7)(0.4)+0.1\\
  &= 0.96-0.28+0.1\\
  &= 0.78.
\end{aligned}
$$

Si usamos la función sigmoide:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

obtenemos:

$$
\sigma(0.78)\approx0.686
$$

La neurona produce una salida cercana a 0.686.

## Funciones de activación

Las funciones de activación permiten que la red represente relaciones no lineales.

### Función escalón

$$
f(z)=
\begin{cases}
1, & z\geq0\\
0, & z<0
\end{cases}
$$

Fue utilizada en los primeros perceptrones.

### Función sigmoide

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Características:

- produce valores entre 0 y 1;
- es útil en clasificación binaria;
- puede interpretarse como probabilidad;
- puede saturarse para valores grandes.

### Tangente hiperbólica

$$
\tanh(z)=\frac{e^z-e^{-z}}{e^z+e^{-z}}
$$

Produce valores entre -1 y 1.

### ReLU

$$
\operatorname{ReLU}(z)=\max(0,z)
$$

Es una de las activaciones más utilizadas en capas ocultas.

### Softmax

Para clasificación multiclase:

$$
P(y=k)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
$$

Convierte las salidas en probabilidades que suman 1.

## El perceptrón

El perceptrón es uno de los modelos neuronales más sencillos.

Calcula:

$$
\widehat{y}=f(\mathbf{w}^{T}\mathbf{x}+b)
$$

y actualiza los pesos cuando comete errores.

La regla básica de actualización es:

$$
w_j^{nuevo}=w_j^{anterior}+\eta(y-\widehat{y})x_j
$$

donde:

- $\eta$ es la tasa de aprendizaje;
- $y$ es la clase real;
- $\widehat{y}$ es la predicción.

### Limitación

Un perceptrón simple solo resuelve problemas linealmente separables. El problema XOR es el ejemplo clásico que requiere una capa oculta.

## Estructura de una red multicapa

Una red neuronal multicapa suele tener:

1. capa de entrada;
2. una o más capas ocultas;
3. capa de salida.

Cada neurona de una capa puede conectarse con las neuronas de la siguiente.

### Capa de entrada

Recibe las variables predictoras.

### Capas ocultas

Aprenden combinaciones intermedias de las características.

### Capa de salida

Produce la predicción final.

Ejemplos:

- una neurona sigmoide para clasificación binaria;
- varias neuronas softmax para clasificación multiclase;
- una neurona lineal para regresión.

## Propagación hacia adelante

La **propagación hacia adelante** calcula la salida de la red a partir de las entradas.

Para una capa oculta:

$$
\mathbf{h}=f(\mathbf{W}^{(1)}\mathbf{x}+\mathbf{b}^{(1)})
$$

Para la salida:

$$
\widehat{\mathbf{y}}=g(\mathbf{W}^{(2)}\mathbf{h}+\mathbf{b}^{(2)})
$$

La red transforma sucesivamente la información hasta producir una predicción.

## Función de pérdida

La función de pérdida mide qué tan lejos está la predicción del valor real.

### Error cuadrático medio

$$
MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat{y}_i)^2
$$

Se usa principalmente en regresión.

### Entropía cruzada binaria

$$
L=-\frac{1}{n}\sum_{i=1}^{n}
\left[
y_i\log(\widehat{p}_i)
+(1-y_i)\log(1-\widehat{p}_i)
\right]
$$

Se utiliza en clasificación binaria.

### Entropía cruzada categórica

$$
L=-\frac{1}{n}\sum_{i=1}^{n}\sum_{k=1}^{K}
y_{ik}\log(\widehat{p}_{ik})
$$

Se utiliza en clasificación multiclase.

## Descenso de gradiente

El entrenamiento busca valores de los pesos que minimicen la pérdida.

La actualización general es:

$$
w^{\mathrm{nuevo}}
=
w^{\mathrm{anterior}}
-
\eta\frac{\partial L}{\partial w}
$$

donde $\eta$ es la tasa de aprendizaje.

### Tasa demasiado pequeña

- aprendizaje lento;
- muchas iteraciones.

### Tasa demasiado grande

- oscilaciones;
- divergencia;
- pérdida inestable.

## Retropropagación

La **retropropagación** calcula cómo contribuye cada peso al error.

El proceso es:

1. realizar propagación hacia adelante;
2. calcular la pérdida;
3. obtener derivadas mediante la regla de la cadena;
4. propagar el error hacia atrás;
5. actualizar pesos y sesgos;
6. repetir.

La retropropagación no es un modelo distinto, sino el mecanismo principal para entrenar redes multicapa.

## Implementación manual de una neurona en R


In [ ]:
sigmoide <- function(z) {
  1 / (1 + exp(-z))
}

x <- c(0.8, 0.4)
w <- c(1.2, -0.7)
b <- 0.1

z <- sum(x * w) + b
salida <- sigmoide(z)

z
salida


## Laboratorio interactivo: construye una neurona artificial

El siguiente laboratorio se ejecuta directamente en tu navegador mediante **Shinylive**. No requiere un servidor Shiny externo.

Puedes modificar las entradas, los pesos, el sesgo, la función de activación y el umbral de clasificación.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


1. Mantén las entradas fijas y cambia solamente el sesgo.
2. Cambia el signo de uno de los pesos.
3. Compara sigmoide, ReLU y tangente hiperbólica.
4. Coloca ambos pesos en cero.
5. Modifica el umbral y observa si cambia la clase.

### Laboratorio interactivo disponible en la versión web

La versión web del capítulo incluye un laboratorio donde pueden modificarse las entradas, los pesos, el sesgo, la función de activación y el umbral de clasificación de una neurona artificial.

## Perceptrón desde cero en R


In [ ]:
entrenar_perceptron <- function(X, y,
                               eta = 0.1,
                               epocas = 100) {

  X <- as.matrix(X)
  y <- as.numeric(y)

  pesos <- rep(0, ncol(X))
  sesgo <- 0

  for (epoca in seq_len(epocas)) {

    for (i in seq_len(nrow(X))) {

      z <- sum(X[i, ] * pesos) + sesgo
      pred <- ifelse(z >= 0, 1, 0)
      error <- y[i] - pred

      pesos <- pesos + eta * error * X[i, ]
      sesgo <- sesgo + eta * error
    }
  }

  list(
    pesos = pesos,
    sesgo = sesgo
  )
}


Ejemplo con la compuerta AND:


In [ ]:
X_and <- data.frame(
  x1 = c(0, 0, 1, 1),
  x2 = c(0, 1, 0, 1)
)

y_and <- c(0, 0, 0, 1)

modelo_and <- entrenar_perceptron(
  X_and,
  y_and,
  eta = 0.1,
  epocas = 20
)

modelo_and


Función de predicción:


In [ ]:
predecir_perceptron <- function(modelo, X) {

  X <- as.matrix(X)

  z <- as.numeric(
    X %*% modelo$pesos +
      modelo$sesgo
  )

  ifelse(z >= 0, 1, 0)
}

predecir_perceptron(
  modelo_and,
  X_and
)


## Ejemplo de XOR

La compuerta XOR produce:

| $x_1$ | $x_2$ | XOR |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

No puede resolverse mediante una sola frontera lineal. Una capa oculta permite construir regiones más complejas.

## Preparación de los datos

Antes de entrenar una red neuronal es recomendable:

- tratar valores faltantes;
- convertir categorías;
- separar entrenamiento, validación y prueba;
- normalizar variables numéricas;
- revisar desbalance;
- eliminar identificadores;
- evitar fuga de información.

## Normalización

Las redes suelen entrenarse mejor cuando las variables tienen escalas comparables.

### Estandarización

$$
z=\frac{x-\bar{x}}{s}
$$

### Mínimo-máximo

$$
x^\ast=\frac{x-\min(x)}{\max(x)-\min(x)}
$$

Los parámetros deben calcularse solo con entrenamiento.


In [ ]:
set.seed(123)

indice <- sample(
  seq_len(nrow(iris)),
  size = round(0.70 * nrow(iris))
)

train <- iris[indice, ]
test <- iris[-indice, ]

medias <- sapply(train[, 1:4], mean)
desvios <- sapply(train[, 1:4], sd)

x_train <- scale(
  train[, 1:4],
  center = medias,
  scale = desvios
)

x_test <- scale(
  test[, 1:4],
  center = medias,
  scale = desvios
)


## Clasificación binaria con neuralnet

Para que la generación del libro sea estable, los bloques que entrenan la red con `neuralnet` se muestran, pero no se ejecutan automáticamente durante el renderizado. Puedes ejecutarlos en RStudio después de instalar el paquete con `PREPARAR_PAQUETES_RED_NEURONAL.bat`.

Crearemos un problema con dos especies de `iris`.


In [ ]:
iris_bin <- subset(
  iris,
  Species != "setosa"
)

iris_bin$clase <- ifelse(
  iris_bin$Species == "virginica",
  1,
  0
)

iris_bin$Species <- NULL


División:


In [ ]:
set.seed(321)

idx <- sample(
  seq_len(nrow(iris_bin)),
  size = round(0.70 * nrow(iris_bin))
)

train_bin <- iris_bin[idx, ]
test_bin <- iris_bin[-idx, ]


Normalización segura:


In [ ]:
variables <- setdiff(
  names(train_bin),
  "clase"
)

medias_bin <- sapply(
  train_bin[, variables],
  mean
)

desvios_bin <- sapply(
  train_bin[, variables],
  sd
)

train_bin[, variables] <- scale(
  train_bin[, variables],
  center = medias_bin,
  scale = desvios_bin
)

test_bin[, variables] <- scale(
  test_bin[, variables],
  center = medias_bin,
  scale = desvios_bin
)


Entrenamiento:


## Predicción con neuralnet


## Métricas binarias


## Selección del umbral

El umbral 0.5 no siempre es el mejor.


## Visualizador interactivo de una red neuronal

Este laboratorio representa gráficamente una arquitectura similar a la que se especifica con el argumento `hidden` de `neuralnet`. Modifica el número de neuronas de las capas ocultas y observa cómo cambian la topología y el número de conexiones.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


El visualizador no entrena el modelo dentro del navegador. Su propósito es mostrar la arquitectura definida mediante `hidden`. Cuando ejecutes `plot(modelo_rna)` en RStudio, `neuralnet` dibujará además los pesos y sesgos aprendidos.

### Visualizador disponible en la versión web

La versión web permite modificar el número de entradas, neuronas ocultas y salidas, y genera dinámicamente el dibujo de la red y el código equivalente para `neuralnet`.

## Arquitectura de la red

Una arquitectura se define por:

- número de capas ocultas;
- neuronas por capa;
- activaciones;
- conexiones;
- regularización.

No existe una arquitectura universal.

### Red muy pequeña

Puede producir subajuste.

### Red demasiado grande

Puede memorizar entrenamiento y sobreajustar.

Una estrategia razonable es comenzar con una red pequeña y aumentar complejidad gradualmente.

## Épocas, lotes y optimizadores

### Época

Una época representa una pasada completa por entrenamiento.

### Lote

Un lote es un subconjunto utilizado para calcular una actualización.

### Descenso por lotes

Usa todo el conjunto.

### Descenso estocástico

Actualiza con una observación.

### Mini-batch

Usa pequeños grupos y es la opción más común.

### Optimizadores

Entre los más conocidos se encuentran:

- SGD;
- Momentum;
- RMSprop;
- Adam.

## Sobreajuste

Una red sobreajustada obtiene:

- error bajo en entrenamiento;
- error alto en validación o prueba.

Señales:

- la pérdida de entrenamiento sigue bajando;
- la pérdida de validación comienza a subir;
- crece la diferencia entre ambas.

## Regularización

### Regularización L2

Agrega una penalización:

$$
L_{total}=L+\lambda\sum_j w_j^2
$$

### Regularización L1

$$
L_{total}=L+\lambda\sum_j |w_j|
$$

### Dropout

Desactiva aleatoriamente una proporción de neuronas durante el entrenamiento.

### Early stopping

Detiene el entrenamiento cuando la validación deja de mejorar.

## Implementación moderna con keras

El paquete `keras` permite construir redes utilizando una interfaz de alto nivel.

La instalación de `keras` y su motor puede variar según el equipo. Conviene ejecutar este ejemplo en un entorno configurado o en Google Colab.


Modelo:


Compilación:


Entrenamiento:


Evaluación:


## Curvas de aprendizaje

Las curvas de pérdida y exactitud ayudan a diagnosticar el entrenamiento.


Debe compararse:

- desempeño de entrenamiento;
- desempeño de validación;
- punto donde comienza el sobreajuste.

## Clasificación multiclase

En un problema con $K$ clases:

- la capa de salida suele tener $K$ neuronas;
- se utiliza softmax;
- la clase predicha es la de mayor probabilidad.

$$
\widehat{y}=
\arg\max_k P(y=k\mid\mathbf{x})
$$

## Codificación de la respuesta

Para clasificación multiclase pueden emplearse:

- enteros de 0 a $K-1$;
- variables indicadoras one-hot.

Ejemplo one-hot:

| Clase | Neurona 1 | Neurona 2 | Neurona 3 |
|---|---:|---:|---:|
| A | 1 | 0 | 0 |
| B | 0 | 1 | 0 |
| C | 0 | 0 | 1 |

## Inicialización de pesos

Todos los pesos no deben comenzar con el mismo valor, porque las neuronas aprenderían exactamente lo mismo.

Las inicializaciones modernas buscan:

- romper simetría;
- mantener estable la varianza;
- mejorar el flujo de gradientes.

## Gradientes que desaparecen o explotan

En redes profundas, los gradientes pueden:

- aproximarse a cero;
- crecer excesivamente.

Consecuencias:

- aprendizaje muy lento;
- inestabilidad;
- pesos extremos.

Soluciones comunes:

- ReLU;
- inicialización adecuada;
- normalización;
- arquitecturas apropiadas;
- control de la tasa de aprendizaje.

## Importancia de la reproducibilidad

Para reproducir resultados:

- fijar semillas;
- registrar arquitectura;
- guardar parámetros;
- documentar divisiones;
- conservar versiones;
- guardar modelos entrenados;
- anotar métricas y umbrales.


In [ ]:
set.seed(2026)


## Interpretabilidad

Las redes neuronales suelen considerarse menos interpretables que:

- regresión logística;
- árboles de decisión;
- modelos lineales.

Herramientas posibles:

- importancia por permutación;
- análisis de sensibilidad;
- gráficos de dependencia;
- SHAP;
- LIME;
- inspección de activaciones.

La interpretación debe acompañarse de validación y conocimiento del dominio.

## Aplicación biomédica

Ejemplo: clasificar pacientes según riesgo de enfermedad.

Variables:

- edad;
- presión arterial;
- glucosa;
- colesterol;
- frecuencia cardiaca.

Respuesta:

- riesgo bajo;
- riesgo alto.

Debe prestarse especial atención a:

- sensibilidad;
- falsos negativos;
- calidad de los datos;
- validación externa;
- sesgo de selección;
- explicabilidad.

## Aplicación ambiental

Ejemplo: clasificar niveles de contaminación.

Variables:

- PM10;
- ozono;
- temperatura;
- humedad;
- velocidad del viento;
- hora;
- estación.

Respuesta:

- aceptable;
- no aceptable.

La red puede capturar relaciones no lineales, pero debe compararse con modelos más simples.

## Comparación con otros algoritmos

| Aspecto | Red neuronal | Regresión logística | Árbol | Random Forest |
|---|---|---|---|---|
| No linealidad | Alta | Limitada | Alta | Alta |
| Interpretación | Baja | Alta | Alta | Media |
| Preparación | Alta | Media | Media | Media |
| Escalamiento | Importante | Recomendable | Poco importante | Poco importante |
| Datos requeridos | Frecuentemente muchos | Menos | Moderados | Moderados |
| Costo | Puede ser alto | Bajo | Bajo | Moderado |

## Ventajas

- modelan relaciones complejas;
- representan no linealidades;
- se adaptan a clasificación y regresión;
- pueden aprender características internas;
- escalan a problemas de gran dimensión;
- son base de aprendizaje profundo.

## Limitaciones

- requieren preparación cuidadosa;
- pueden necesitar muchos datos;
- implican varios hiperparámetros;
- pueden sobreajustar;
- son menos interpretables;
- el entrenamiento puede ser costoso;
- los resultados dependen de arquitectura e inicialización.

## Errores frecuentes

1. No normalizar variables.
2. Usar el conjunto de prueba para ajustar.
3. Entrenar demasiadas épocas sin validación.
4. Usar una red enorme para pocos datos.
5. Evaluar solo exactitud.
6. Ignorar el desbalance.
7. No fijar semilla.
8. No guardar la arquitectura.
9. Interpretar probabilidades como certezas.
10. Comparar modelos con particiones diferentes.
11. Dejar identificadores como predictores.
12. No comparar contra un modelo simple.

## Flujo de trabajo recomendado

1. Definir el objetivo.
2. Preparar los datos.
3. Separar entrenamiento, validación y prueba.
4. Normalizar con entrenamiento.
5. Construir una arquitectura pequeña.
6. Entrenar y revisar curvas.
7. Ajustar hiperparámetros.
8. Aplicar regularización.
9. Evaluar en prueba una sola vez.
10. Comparar con modelos base.
11. Interpretar errores.
12. Documentar y guardar el modelo.

## Actividad guiada

Utiliza `iris` para clasificación multiclase.

1. Divide los datos en entrenamiento y prueba.
2. Normaliza las cuatro variables.
3. Diseña una red con una capa oculta.
4. Entrena el modelo.
5. Obtén probabilidades.
6. Asigna la clase más probable.
7. Construye la matriz de confusión.
8. Calcula exactitud.
9. Repite con más neuronas.
10. Compara los resultados.
11. Describe si existe sobreajuste.
12. Compara contra k-NN o Random Forest.

## Ejercicios

### Ejercicio 1

Calcula manualmente la salida de una neurona sigmoide.

### Ejercicio 2

Programa una neurona ReLU en R.

### Ejercicio 3

Entrena un perceptrón para la compuerta OR.

### Ejercicio 4

Explica por qué XOR requiere una capa oculta.

### Ejercicio 5

Entrena una red binaria con `neuralnet`.

### Ejercicio 6

Compara arquitecturas `hidden = 3`, `hidden = 5` y `hidden = c(5,3)`.

### Ejercicio 7

Modifica el umbral de clasificación y analiza sensibilidad y especificidad.

### Ejercicio 8

Construye una red multiclase con `keras`.

### Ejercicio 9

Compara la red con regresión logística.

### Ejercicio 10

Describe tres medidas para reducir sobreajuste.

## Preguntas de reflexión

- ¿Qué función cumple el sesgo?
- ¿Por qué se necesitan activaciones no lineales?
- ¿Qué diferencia existe entre propagación hacia adelante y retropropagación?
- ¿Por qué debe separarse validación de prueba?
- ¿Qué indica una pérdida de validación creciente?
- ¿Cuándo conviene usar una red neuronal?
- ¿Cuándo sería preferible un modelo más simple?
- ¿Por qué las probabilidades deben interpretarse con cautela?

## Caso aplicado B: red neuronal con COVID-19

En este ejemplo entrenamos una red neuronal pequeña para clasificar `MURIO` con datos de COVID-19 México 2022. Se utiliza una arquitectura deliberadamente sencilla para que el objetivo sea comprender el procedimiento y no construir un sistema clínico.

> **Uso académico:** la salida de la red es una predicción estadística sobre esta muestra y no un pronóstico médico individual.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_nn <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
                  OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_nn <- covid_nn |>
    dplyr::sample_n(min(8000, nrow(covid_nn)))

  set.seed(2026)
  idx_nn_covid <- unlist(lapply(
    split(seq_len(nrow(covid_nn)), covid_nn$MURIO),
    function(i) sample(i, floor(0.80 * length(i)))
  ))

  train_nn_covid <- covid_nn[idx_nn_covid, ]
  test_nn_covid <- covid_nn[-idx_nn_covid, ]

  predictores_nn <- c("EDAD", "NEUMONIA", "DIABETES", "HIPERTENSION",
                      "OBESIDAD", "RENAL_CRONICA", "NUM_COMORBILIDADES")

  medias_nn <- sapply(train_nn_covid[predictores_nn], mean)
  desv_nn <- sapply(train_nn_covid[predictores_nn], sd)
  desv_nn[desv_nn == 0] <- 1

  train_nn_covid[predictores_nn] <- scale(
    train_nn_covid[predictores_nn], center = medias_nn, scale = desv_nn
  )
  test_nn_covid[predictores_nn] <- scale(
    test_nn_covid[predictores_nn], center = medias_nn, scale = desv_nn
  )
}


La estandarización se calcula exclusivamente con el conjunto de entrenamiento y después se aplica al conjunto de prueba. Esto evita utilizar información de prueba durante el aprendizaje.


In [ ]:
if (exists("train_nn_covid")) {
  set.seed(2026)
  modelo_nn_covid <- neuralnet::neuralnet(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_nn_covid,
    hidden = c(5, 3),
    linear.output = FALSE,
    lifesign = "none",
    stepmax = 1e6
  )

  prob_nn_covid <- as.numeric(
    neuralnet::compute(
      modelo_nn_covid,
      test_nn_covid[predictores_nn]
    )$net.result[, 1]
  )

  pred_nn_covid <- ifelse(prob_nn_covid >= 0.50, 1, 0)
  matriz_nn_covid <- table(
    Real = factor(test_nn_covid$MURIO, levels = c(0,1), labels = c("Sin defunción", "Defunción")),
    Predicho = factor(pred_nn_covid, levels = c(0,1), labels = c("Sin defunción", "Defunción"))
  )
  matriz_nn_covid
}


In [ ]:
if (exists("matriz_nn_covid")) {
  VP <- matriz_nn_covid["Defunción", "Defunción"]
  FN <- matriz_nn_covid["Defunción", "Sin defunción"]
  FP <- matriz_nn_covid["Sin defunción", "Defunción"]
  VN <- matriz_nn_covid["Sin defunción", "Sin defunción"]
  div <- function(a,b) ifelse(b == 0, NA_real_, as.numeric(a/b))
  data.frame(
    exactitud = div(VP+VN, sum(matriz_nn_covid)),
    sensibilidad = div(VP, VP+FN),
    especificidad = div(VN, VN+FP)
  )
}


La red puede representar relaciones no lineales entre edad y comorbilidades, pero eso no garantiza que supere a modelos más simples. La comparación con regresión logística, k-NN, árboles, Random Forest, SVM y Naive Bayes será más importante que observar una sola métrica.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=gk3bt0h3WUU) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pdf) | [Descargar PDF](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pdf){download="capitulo-12-redes-neuronales-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pptx){download="capitulo-12-redes-neuronales-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png) | [Descargar PNG](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png){download="capitulo-12-redes-neuronales-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/12-redes-neuronales.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 12](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png)](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/12-redes-neuronales.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=gk3bt0h3WUU>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Conclusión

Las redes neuronales artificiales combinan neuronas simples para aprender relaciones complejas.

Sus componentes fundamentales son:

- entradas;
- pesos;
- sesgos;
- funciones de activación;
- capas;
- función de pérdida;
- optimización;
- retropropagación.

Una red efectiva no depende solamente de aumentar capas o neuronas. También requiere:

- datos de calidad;
- normalización correcta;
- validación;
- regularización;
- selección cuidadosa de arquitectura;
- evaluación con métricas apropiadas;
- comparación con modelos más sencillos.

En el siguiente capítulo podremos abordar métodos no supervisados, comenzando con **agrupamiento k-means**.
